# Phase 2 - PyCaret Classification on 150k Papers

**Purpose**: Run PyCaret metadata classifier on full V5.1 dataset (39,780 papers)

**Script**: `scripts/06b_run_phase2_pycaret_classification.py`

**Environment**: Google Colab (CPU only - PyCaret runs efficiently on CPU)

**Runtime**: ~10-15 minutes (CPU)

---

## Usage

1. **Runtime**: Default CPU runtime is sufficient (no GPU needed)
2. **Execute cells**: Run cells in order from top to bottom
3. **Monitor**: Watch progress in cell outputs
4. **Results**: Automatically saved to Google Drive

---

In [ ]:
# =============================================================================
# CELL 1: MOUNT GOOGLE DRIVE
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("📦 Phase 2 files are now accessible")
print("🔗 Ready for PyCaret classification on 150k papers")

In [ ]:
# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================
import os
from datetime import datetime

# =============================================================================
# MODE SELECTION
# =============================================================================
TEST_MODE = False  # Set to True for quick testing (~30 sec with 100 papers)
                   # Set to False for full run (~10-15 min with 39,780 papers)

# Session Management
import random
import string

SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(f"\n🔖 Session ID: {SESSION_ID}")
print(f"📅 Timestamp: {TIMESTAMP}")

# =============================================================================
# PATH CONFIGURATION
# =============================================================================
BASE_PATH = '/content/drive/MyDrive'
REPO_NAME = 'inventory_2022'
REPO_PATH = os.path.join(BASE_PATH, REPO_NAME)

if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(f"❌ Repository not found at: {REPO_PATH}")

VALIDATION_PATH = os.path.join(REPO_PATH, 'validation_spacy_v_BERT')

if not os.path.exists(VALIDATION_PATH):
    raise FileNotFoundError(f"❌ Validation directory not found at: {VALIDATION_PATH}")

print(f"✓ Repository found: {REPO_PATH}")
print(f"✓ Validation directory: {VALIDATION_PATH}")

os.chdir(VALIDATION_PATH)

os.environ['TEST_MODE'] = str(TEST_MODE)
os.environ['SESSION_ID'] = SESSION_ID

if TEST_MODE:
    print("\n🧪 TEST MODE ENABLED")
    print("   ⏱️  Expected runtime: ~30 seconds")
    print("   📊 Dataset: 100 papers (test subset)")
else:
    print("\n🚀 PRODUCTION MODE ENABLED")
    print("   ⏱️  Expected runtime: ~10-15 minutes")
    print("   📊 Dataset: 39,780 papers (full V5.1)")

print(f"\n📁 Working directory: {os.getcwd()}")
print("\n" + "="*60)
print("CONFIGURATION LOADED")
print("="*60)

In [ ]:
# =============================================================================
# CELL 3: INSTALL DEPENDENCIES
# =============================================================================

print("🔧 Installing PyCaret and dependencies...")

# Install PyCaret with specific dependency versions that work in Colab
!pip install --upgrade pip -q
!pip install pycaret[full]==3.0.4 -q

# Verify installation
import warnings
warnings.filterwarnings('ignore')

try:
    from pycaret.classification import load_model, predict_model
    print("\n✅ PyCaret 3.0.4 imported successfully")
except ImportError as e:
    print(f"\n❌ PyCaret import failed: {e}")
    print("Installing with alternative method...")
    !pip install pycaret==3.0.4 --no-deps -q
    !pip install pandas scikit-learn joblib -q
    from pycaret.classification import load_model, predict_model
    print("✅ PyCaret installed with alternative method")

print("\n✅ Environment setup complete")

In [ ]:
# =============================================================================
# CELL 4: VERIFY CPU ENVIRONMENT
# =============================================================================

import numpy as np
import pandas as pd

print("🔍 CPU Environment Check:")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Check CPU info
!cat /proc/cpuinfo | grep "model name" | head -1
!cat /proc/cpuinfo | grep "processor" | wc -l

print("\n✅ CPU environment verified")
print("ℹ️  PyCaret uses CPU-based models (no GPU needed)")
print("⚡ Expected performance: ~10-15 min for 39,780 papers")

In [ ]:
# =============================================================================
# CELL 5: VERIFY INPUT FILES
# =============================================================================

import sys
from pathlib import Path

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

src_path = str(Path(REPO_PATH) / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("🔍 Verifying input files...\n")

# Check Phase 2 input file (V5.1 with EPMC metadata)
input_file = Path(VALIDATION_PATH) / "data/v5.1_cleaned.csv"

if input_file.exists():
    print(f"✓ Phase 2 input: {input_file}")
    print(f"  Size: {input_file.stat().st_size / 1024 / 1024:.1f} MB")
else:
    raise FileNotFoundError(f"Required file not found: {input_file}")

# Check PyCaret model
model_path = Path(REPO_PATH) / "pycaret_models/test_mode_true/pycaret_metadata_classifier_v1.pkl"

if model_path.exists():
    print(f"\n✓ PyCaret model: {model_path}")
    print(f"  Size: {model_path.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"  Features: 92 (metadata-based)")
else:
    raise FileNotFoundError(f"Required model not found: {model_path}")

# Check script
script_path = Path(VALIDATION_PATH) / "scripts/06b_run_phase2_pycaret_classification.py"
if not script_path.exists():
    raise FileNotFoundError(f"Required script not found: {script_path}")
print(f"\n✓ Script verified: {script_path}")

print("\n✅ All required files verified")

In [ ]:
# =============================================================================
# CELL 6: RUN PYCARET CLASSIFICATION
# =============================================================================

from pathlib import Path

print("🚀 Running PyCaret Classification on 150k papers...\n")
print("="*70)

script_path = Path(VALIDATION_PATH) / "scripts/06b_run_phase2_pycaret_classification.py"
%run {script_path}

print("\n" + "="*70)
print("✅ PyCaret Classification Complete")

In [ ]:
# =============================================================================
# CELL 7: DISPLAY RESULTS
# =============================================================================
import pandas as pd
from pathlib import Path

TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")

print("📊 Classification Results Summary\n")
print("="*70)

results_file = Path(VALIDATION_PATH) / f"results/phase2/classification/pycaret_classification_150k{output_suffix}.csv"

if results_file.exists():
    df_results = pd.read_csv(results_file)
    
    print(f"\n📄 Results file: {results_file}")
    print(f"📦 File size: {results_file.stat().st_size / 1024 / 1024:.1f} MB")
    
    print(f"\n📊 Classification Statistics:")
    print(f"   Total papers: {len(df_results):,}")
    
    if 'predicted_label' in df_results.columns:
        bio_count = (df_results['predicted_label'] == 1).sum()
        not_bio_count = (df_results['predicted_label'] == 0).sum()
        
        print(f"   Predicted bio-resource: {bio_count:,} ({100*bio_count/len(df_results):.1f}%)")
        print(f"   Predicted not-bio-resource: {not_bio_count:,} ({100*not_bio_count/len(df_results):.1f}%)")
        
        if 'prediction_score' in df_results.columns:
            pos_scores = df_results[df_results['predicted_label'] == 1]['prediction_score']
            if len(pos_scores) > 0:
                print(f"\n📈 Prediction Confidence (Positive Class):")
                print(f"   Mean: {pos_scores.mean():.3f}")
                print(f"   Median: {pos_scores.median():.3f}")
                print(f"   Min: {pos_scores.min():.3f}")
                print(f"   Max: {pos_scores.max():.3f}")
    
    print(f"\n📋 Sample Results (first 5):")
    display_cols = ['id', 'predicted_label']
    if 'prediction_score' in df_results.columns:
        display_cols.append('prediction_score')
    
    print(df_results[display_cols].head())
    
    print(f"\n✅ Results successfully saved to Drive")
    print(f"   Location: {results_file}")
    
else:
    print(f"⚠️ Results file not found: {results_file}")

print("\n" + "="*70)

In [ ]:
# =============================================================================
# CELL 8: COMPLETION SUMMARY
# =============================================================================

from datetime import datetime
from pathlib import Path

TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")
results_file = Path(VALIDATION_PATH) / f"results/phase2/classification/pycaret_classification_150k{output_suffix}.csv"

print("\n" + "🎉" * 35)
print("🎉 PYCARET CLASSIFICATION COMPLETE")
print("🎉" * 35)

print(f"\n📊 Session Information:")
print(f"   Session ID: {SESSION_ID if SESSION_ID else 'N/A'}")
print(f"   Mode: {'TEST' if TEST_MODE else 'PRODUCTION'}")
print(f"   Completion Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📁 Output Files:")
if results_file.exists():
    print(f"   📄 {results_file}")
else:
    print(f"   ⚠️ Results file not created (check errors above)")

print(f"\n🎯 Next Steps:")
print(f"   1. Review results above")
print(f"   2. Compare with V2 BERT classification results")
print(f"   3. Run NER on classified positives from either model")

print(f"\n🏁 Notebook execution complete!")
print(f"📊 Results are automatically saved to Google Drive")